# Synchronized audio-video generation with LTX-2.3 and OpenVINO(TM)

[LTX-2.3](https://huggingface.co/diffusers/LTX-2.3-Diffusers) is a diffusion transformer model that generates video and synchronized audio in a single pipeline. The prompt can describe both the visual scene and its soundtrack, including ambient sounds, music, and speech.

<video src="https://github.com/user-attachments/assets/0a297ba9-579d-4867-8e9d-b65eaead7ad1" controls width="512"></video>

*Example from the original notebook: "A cat stretches lazily on a sunny windowsill, purring softly, while birds chirp outside." with the INT8 model on CPU (768x512, 121 frames, with audio).*

This tutorial shows how to export LTX-2.3 to OpenVINO Intermediate Representation with [Optimum Intel](https://huggingface.co/docs/optimum/intel/index), compress its weights with [NNCF](https://github.com/openvinotoolkit/nncf), and run one of two workflows:

- **Text-to-audio-video** generates video and audio from a text prompt. It can run with [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) `Text2VideoPipeline` or with the Optimum Intel pipeline.
- **Image-to-audio-video** uses an initial image together with a text prompt. It runs with the Optimum Intel pipeline.

A single text-to-video export is reused by both workflows. The exporter also includes the VAE encoder needed for image conditioning; the text-to-video pipelines ignore that extra component.

LTX-2.3 is a large model. Download, export, compression, and inference require substantial disk space and memory.

Model weights are subject to the [LTX-2 Community License Agreement](https://github.com/Lightricks/LTX-2/blob/main/LICENSE-2). Review its terms before use.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Compress model weights](#Compress-model-weights)
- [Run audio-video generation](#Run-audio-video-generation)
- [Interactive inference](#Interactive-inference)
- [Limitations](#Limitations)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ltx2.3-audio-video/ltx2.3-audio-video.ipynb" />

## Prerequisites
[back to top](#Table-of-contents:)

This notebook uses `diffusers/LTX-2.3-Diffusers`, the full single-stage checkpoint, not the distilled model or LTX-2.5. The following components are needed together:

| Component | Support used by this notebook |
| --- | --- |
| Optimum Intel | A pinned revision of `main` with [external RoPE inputs](https://github.com/huggingface/optimum-intel/pull/2070), the [group-wise INT8 recipe](https://github.com/huggingface/optimum-intel/pull/2076), and the [updated INT4 recipe](https://github.com/huggingface/optimum-intel/pull/2127). These updates are not in the PyPI 2.2.0 release. |
| OpenVINO GenAI | Nightly packages with [LTX-2.3 text-to-audio-video](https://github.com/openvinotoolkit/openvino.genai/pull/4559) and [external RoPE support](https://github.com/openvinotoolkit/openvino.genai/pull/4692). The PyPI 2026.4.1 release does not include them. LTX-2.3 image-to-video still uses Optimum Intel. |
| OpenVINO and OpenVINO Tokenizers | Installed together with GenAI from the nightly index to resolve a compatible runtime, PyTorch frontend, device plugins, and tokenizer extension. CPU and GPU are selectable; full-model execution on these devices is separate from upstream tiny-model tests. |
| NNCF | A pinned development commit containing the [merged group-wise INT8 support](https://github.com/openvinotoolkit/nncf/pull/4278), which is not in PyPI 3.4.0. |
| Diffusers and Transformers | The notebook keeps Diffusers 0.40.0 and Transformers 5.10.4 for the LTX-2.3 API used below. Newer Diffusers versions can require a different Transformers version. |

The runtime requirements stay within the 2026.5 series (`>=2026.5.0.dev20261116,<2026.6`), accepting a future stable 2026.5.0 release without selecting the 2027.0 nightlies. Matching 2026.5.1 nightly packages from November 16, 2026 are currently available. NNCF is installed before Optimum Intel, and Optimum Intel is resolved together with the runtime packages to avoid first installing and then replacing their older PyPI versions.

The recommended INT8 path calls NNCF directly. Optimum Intel now contains the same recipe, but `optimum-cli --weight-format int8` selects it only with NNCF **3.5.0 or newer**. Even a development build containing the merged change is below that release version, so the CLI would instead select its older per-channel recipe. INT4 uses the current model-specific CLI defaults.

In [ ]:
import urllib.request
from pathlib import Path

helper_files = ["notebook_utils.py", "cmd_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder.
setup_notebook_venv()

In [ ]:
%pip install -qU "setuptools>=77.0" "wheel"
%pip install -qU "torch==2.10" "torchvision==0.25" "diffusers==0.40.0" "transformers==5.10.4" "accelerate" "sentencepiece" "protobuf>=3.20" "huggingface-hub" "av" "ipywidgets" "gradio>=6.0" --extra-index-url https://download.pytorch.org/whl/cpu
# Merged group-wise INT8 support: https://github.com/openvinotoolkit/nncf/pull/4278
# Use the updated build tools above, not an older setuptools inherited from Jupyter.
%pip install -q --no-build-isolation "nncf @ https://github.com/openvinotoolkit/nncf/archive/cc1a670b654f7688403b62e3dd7b619a54e82980.tar.gz"
%pip install -qU --pre "git+https://github.com/huggingface/optimum-intel.git@883580d3de55f8f54d857eda5a3dfed44e4c06d9" "openvino>=2026.5.0.dev20261116,<2026.6" "openvino-tokenizers>=2026.5.0.0.dev20261116,<2026.6" "openvino-genai>=2026.5.0.0.dev20261116,<2026.6" --extra-index-url https://download.pytorch.org/whl/cpu --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

In [ ]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("ltx2.3-audio-video.ipynb")

In [ ]:
if not Path("gradio_helper.py").is_file():
    url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/ltx2.3-audio-video/gradio_helper.py"
    with urllib.request.urlopen(url, timeout=10) as response:  # nosec B310 - fixed HTTPS repository URL
        Path("gradio_helper.py").write_bytes(response.read())

from gradio_helper import DEFAULT_PROMPTS, format_perf_metrics, get_guidance_config, validate_static_config

## Convert the model to OpenVINO IR
[back to top](#Table-of-contents:)

Optimum Intel's LTX-2.3 exporter produces the text encoder, connectors, transformer, video VAE encoder and decoder, audio VAE decoder, and vocoder for either task. Export the **text-to-video** task so that `model_index.json` identifies `LTX2Pipeline`, the class accepted by GenAI. This export still contains the VAE encoder and can also be loaded explicitly as `OVLTX2ImageToVideoPipeline` for image conditioning.

The PyTorch checkpoint (about 90 GB) is downloaded first with a progress bar. The export itself prints no progress and can take a long time.

The following weight formats are available. They describe stored weights, not the precision of every operation or output.

| Weight format | Transformer and connectors | Text encoder |
| --- | --- | --- |
| FP16 | FP16, with sensitive parameters retained in FP32 by the exporter | FP16 |
| INT8 (recommended) | INT8 symmetric, group size 32, sensitive layers excluded from compression | INT8 asymmetric, per-channel |
| INT4 (experimental) | INT4 symmetric, group size 32, including connectors | INT4 asymmetric, group size 128 |

The recommended INT8 scheme follows the [upstream layer selection](https://github.com/huggingface/optimum-intel/pull/2076). Until NNCF 3.5.0 is released, the notebook exports FP16 first and then compresses the transformer, connectors, and text encoder directly with NNCF in the [next section](#Compress-model-weights), avoiding the CLI version fallback described above. INT4 is exported directly with `optimum-cli`, without extra compression arguments that would override its model-specific defaults.

The current exporter computes RoPE outside the transformer and connectors, avoiding large-angle calculations in FP16 on GPU. Exports are saved under a new `ltx-2.3-ov-external-rope` directory rather than reusing earlier IRs. Previous compression similarity measurements do not validate these updated exports; compare quality against FP16 for your prompts before choosing INT4.

In [ ]:
import ipywidgets as widgets

weight_format_widget = widgets.Dropdown(
    options=[
        ("FP16", "fp16"),
        ("INT8 (recommended)", "int8"),
        ("INT4 (experimental; reduced quality)", "int4"),
    ],
    value="int8",
    description="Weights:",
    style={"description_width": "initial"},
)

weight_format_widget

In [ ]:
from cmd_helper import optimum_cli
from diffusers import DiffusionPipeline

model_id = "diffusers/LTX-2.3-Diffusers"
weight_format = weight_format_widget.value
models_dir = Path("ltx-2.3-ov-external-rope")
fp16_model_path = models_dir / "FP16"

if weight_format == "int8":
    # INT8 is produced from the FP16 export by NNCF in the next section.
    export_format, export_path = "fp16", fp16_model_path
    model_path = models_dir / "INT8-NNCF"
else:
    export_format, export_path = weight_format, models_dir / weight_format.upper()
    model_path = export_path

if not export_path.exists():
    # optimum-cli runs without console output, so the checkpoint is downloaded to the Hugging Face cache here to show progress.
    DiffusionPipeline.download(model_id)
    optimum_cli(model_id, export_path, additional_args={"task": "text-to-video", "weight-format": export_format})

print(f"Exported {export_format.upper()} model to {export_path}")

## Compress model weights
[back to top](#Table-of-contents:)

This step runs only for the INT8 weight format. [NNCF](https://github.com/openvinotoolkit/nncf) `compress_weights` is applied to three OpenVINO models of the FP16 export:

- **Transformer** and **connectors** use symmetric INT8 with a group size of 32. Input and output projections, timestep and prompt modulation, and audio-video cross-attention modulation layers are excluded from compression because they are the most sensitive to quantization.
- **Text encoder** uses asymmetric per-channel INT8.

Group-wise INT8 requires the NNCF build installed in the [Prerequisites](#Prerequisites) section. NNCF 3.4.0 accepts only per-channel INT8 and raises `ParameterNotSupportedError` for `group_size=32`; the merged development commit supports this call directly.

The remaining components (VAE encoder and decoders, audio VAE, vocoder) are small and are copied without changes. The compressed model is saved into a separate directory, so the FP16 export is kept and can be removed afterwards to free disk space. Compressing the transformer requires enough RAM to hold its FP16 weights.

Existing output directories are reused. If you created INT8 output with an earlier notebook revision, regenerate it from the preserved FP16 export to apply the updated serialization that retains sensitive FP32 constants.

In [ ]:
import gc
import shutil

import nncf
import openvino as ov
from nncf import CompressWeightsMode, IgnoredScope

int8_compression = {
    "transformer": dict(
        mode=CompressWeightsMode.INT8_SYM,
        group_size=32,
        # Patterns are matched with re.match against node names, hence the leading ".*".
        ignored_scope=IgnoredScope(
            patterns=[
                r".*proj_in.*",  # patchify_proj, audio_patchify_proj
                r".*proj_out.*",  # proj_out, audio_proj_out
                r".*time_embed.*",  # adaln_single, audio_adaln_single
                r".*av_cross_attn_.*",  # av_ca_*_adaln_single
                r".*prompt_adaln.*",  # prompt_adaln_single, audio_prompt_adaln_single
            ]
        ),
    ),
    "connectors": dict(
        mode=CompressWeightsMode.INT8_SYM,
        group_size=32,
        ignored_scope=IgnoredScope(patterns=[r".*text_proj_in"]),
    ),
    "text_encoder": dict(mode=CompressWeightsMode.INT8_ASYM, group_size=-1),
}


def compress_pipeline(src_dir: Path, dst_dir: Path, compression: dict):
    tmp_dir = dst_dir.with_name(dst_dir.name + "-tmp")
    shutil.rmtree(tmp_dir, ignore_errors=True)

    def skip_compressed_models(directory, files):
        if Path(directory).parent == src_dir and Path(directory).name in compression:
            return [file for file in files if file.startswith("openvino_model.")]
        return []

    shutil.copytree(src_dir, tmp_dir, ignore=skip_compressed_models)

    core = ov.Core()
    for name, params in compression.items():
        print(f"Compressing {name}")
        model = core.read_model(src_dir / name / "openvino_model.xml")
        model = nncf.compress_weights(model, **params)
        ov.save_model(model, tmp_dir / name / "openvino_model.xml", compress_to_fp16=False)
        del model
        gc.collect()

    tmp_dir.rename(dst_dir)


if weight_format == "int8" and not model_path.exists():
    compress_pipeline(fp16_model_path, model_path, int8_compression)

print(f"Using {weight_format.upper()} model from {model_path}")

## Run audio-video generation
[back to top](#Table-of-contents:)

Select the generation workflow, the inference backend, and OpenVINO devices. Both workflows load the same exported model directory. CPU is the default; AUTO is excluded so that choosing a GPU uses stage-by-stage compilation in GenAI.

- **OpenVINO GenAI** `Text2VideoPipeline` runs the whole text-to-audio-video pipeline in C++ and reads the LTX-2.3 generation defaults from the exported model. It does not support LTX-2.3 image-to-video yet.
- **Optimum Intel** `OVLTX2Pipeline` and `OVLTX2ImageToVideoPipeline` keep the Diffusers pipeline API and support both workflows.

Image-to-audio-video always uses Optimum Intel. NPU is excluded until the complete LTX-2.3 pipeline, including its audio components, is validated on that device.

For **OpenVINO GenAI**, optionally split the stages across devices: **text encoding** includes Gemma 3 and the connectors, **denoising** uses the transformer, and **decoding** includes the video VAE, audio VAE, and vocoder. For example, select CPU / GPU / CPU to keep the text encoder and decoders off the GPU. The precision hint applies to the denoising device and any other stage assigned to that same device. Optimum Intel uses the single-device selector.

**Static shapes** calls `reshape()` before `compile()` for the clip dimensions and frame rate defined below. Optimum Intel keeps some token-related dimensions dynamic and its static path requires CFG. **Compiled-model caching** sets `CACHE_DIR` for either backend. Static Gradio controls keep the compiled clip dimensions fixed; reload the pipeline to change them, the frame rate, or the CFG batch size.

On GPU, the current GenAI pipeline compiles and releases components by stage instead of keeping all compiled components on the device at once. Host models remain in RAM. This does not guarantee that the full checkpoint fits on every GPU; upstream tiny-model tests are not full-checkpoint memory or quality validation.

In [ ]:
from notebook_utils import device_widget

generation_mode_widget = widgets.Dropdown(
    options=[
        ("Text-to-audio-video", "text-to-video"),
        ("Image-to-audio-video", "image-to-video"),
    ],
    value="text-to-video",
    description="Mode:",
)
backend_widget = widgets.Dropdown(
    options=[
        ("OpenVINO GenAI", "genai"),
        ("Optimum Intel", "optimum"),
    ],
    value="genai",
    description="Backend:",
)
device = device_widget(default="CPU", exclude=["NPU", "AUTO"])
split_devices = widgets.Checkbox(value=False, description="Split GenAI stages across devices", indent=False)
text_encode_device = device_widget(default="CPU", exclude=["NPU", "AUTO"], description="Text + connectors:")
denoise_device = device_widget(default="CPU", exclude=["NPU", "AUTO"], description="Transformer:")
vae_device = device_widget(default="CPU", exclude=["NPU", "AUTO"], description="Video/audio decoders:")
denoise_precision = widgets.Dropdown(options=[("Device default", "default"), ("FP16", "f16"), ("FP32", "f32")], description="Denoise precision:")
static_shapes = widgets.Checkbox(value=False, description="Static shapes", indent=False)
cache_models = widgets.Checkbox(value=True, description="Cache compiled models", indent=False)


def update_device_controls(change=None):
    genai_selected = generation_mode_widget.value == "text-to-video" and backend_widget.value == "genai"
    split_devices.disabled = not genai_selected
    device.disabled = genai_selected and split_devices.value
    denoise_precision.disabled = not genai_selected
    for stage_device in (text_encode_device, denoise_device, vae_device):
        stage_device.disabled = not (genai_selected and split_devices.value)


for control in (generation_mode_widget, backend_widget, split_devices):
    control.observe(update_device_controls, names="value")
update_device_controls()
display(generation_mode_widget, backend_widget, device, split_devices, text_encode_device, denoise_device, vae_device)
display(denoise_precision, static_shapes, cache_models)

The default settings below follow the LTX-2.3 reference example and are the same for both backends, except that the clip is shortened to 49 frames (about 2 seconds at 24 fps) to keep the generation time reasonable. The reference example uses 121 frames; the number of frames must be a multiple of 8 plus 1. With the same seed, the two backends produce close but not bit-exact results. The prompt should describe both what is visible and what should be heard. For image-to-audio-video, provide a URL or local path to the conditioning image. Its default road image has a matching traffic prompt; update the prompt when using a different image.

In [ ]:
from diffusers.pipelines.ltx2.utils import DEFAULT_NEGATIVE_PROMPT

frame_rate = 24.0
seed = 42
generation_settings = {
    "negative_prompt": DEFAULT_NEGATIVE_PROMPT,
    "width": 768,
    "height": 512,
    "num_frames": 49,
    "frame_rate": frame_rate,
    "num_inference_steps": 30,
}

prompt_widget = widgets.Textarea(
    value=DEFAULT_PROMPTS[generation_mode_widget.value],
    description="Prompt:",
    layout=widgets.Layout(width="90%", height="100px"),
    style={"description_width": "initial"},
)
image_widget = widgets.Text(
    value="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cosmos/cosmos-video2world-input.jpg",
    description="Image URL/path:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"},
)

video_cfg = widgets.FloatSlider(value=3.0, min=1.0, max=12.0, step=0.1, description="Video CFG:")
audio_cfg = widgets.FloatSlider(value=7.0, min=1.0, max=15.0, step=0.1, description="Audio CFG:")
fast_mode = widgets.Checkbox(value=False, description="Fast mode (CFG only)", indent=False)


def update_prompt(change):
    if prompt_widget.value in DEFAULT_PROMPTS.values():
        prompt_widget.value = DEFAULT_PROMPTS[change["new"]]


generation_mode_widget.observe(update_prompt, names="value")
display(prompt_widget, video_cfg, audio_cfg, fast_mode)
if generation_mode_widget.value == "image-to-video":
    display(image_widget)

In [ ]:
generation_mode = generation_mode_widget.value
use_genai = generation_mode == "text-to-video" and backend_widget.value == "genai"
generation_args = {**generation_settings, **get_guidance_config(video_cfg.value, audio_cfg.value, fast_mode.value)}
static_config = generation_args.copy() if static_shapes.value else None
compile_config = {"CACHE_DIR": str(models_dir / "compiled_cache")} if cache_models.value else {}

if use_genai:
    import openvino_genai as ov_genai

    pipe = ov_genai.Text2VideoPipeline(model_path)
    config = pipe.get_generation_config()
    config.audio_guidance_scale = generation_args["audio_guidance_scale"]
    config.frame_rate = generation_args["frame_rate"]
    pipe.set_generation_config(config)
    if static_config is not None:
        pipe.reshape(1, generation_args["num_frames"], generation_args["height"], generation_args["width"], generation_args["guidance_scale"])
    denoise_device_name = denoise_device.value if split_devices.value else device.value
    if denoise_precision.value != "default":
        compile_config["DEVICE_PROPERTIES"] = {denoise_device_name: {"INFERENCE_PRECISION_HINT": denoise_precision.value}}
    if split_devices.value:
        pipe.compile(text_encode_device.value, denoise_device.value, vae_device.value, config=compile_config)
    else:
        pipe.compile(device.value, config=compile_config)
else:
    from optimum.intel.openvino import OVLTX2ImageToVideoPipeline, OVLTX2Pipeline

    if generation_mode == "image-to-video" and backend_widget.value == "genai":
        print("OpenVINO GenAI does not support LTX-2.3 image-to-video yet, using Optimum Intel instead.")
    if static_config is not None and max(generation_args["guidance_scale"], generation_args["audio_guidance_scale"]) <= 1.0:
        raise ValueError("Optimum Intel static shapes require CFG: keep at least one CFG scale above 1, or use dynamic shapes.")
    pipeline_class = OVLTX2Pipeline if generation_mode == "text-to-video" else OVLTX2ImageToVideoPipeline
    pipe = pipeline_class.from_pretrained(model_path, device=device.value, compile=False, ov_config=compile_config)
    if static_config is not None:
        pipe.reshape(
            batch_size=1, height=generation_args["height"], width=generation_args["width"], num_images_per_prompt=1, num_frames=generation_args["num_frames"]
        )
    pipe.compile()

In [ ]:
import numpy as np
import torch
from diffusers.utils import encode_video, load_image
from tqdm.auto import tqdm

generation_args = {**generation_settings, **get_guidance_config(video_cfg.value, audio_cfg.value, fast_mode.value)}
validate_static_config(generation_args, static_config)

if use_genai:
    with tqdm(total=generation_args["num_inference_steps"], desc="Denoising") as progress_bar:

        def callback(step, num_steps, latent):
            progress_bar.update(1)
            return False

        result = pipe.generate(prompt_widget.value, generator=ov_genai.TorchGenerator(seed), callback=callback, **generation_args)
    # GenAI returns uint8 video [batch, frames, height, width, channels] and audio [batch, channels, samples].
    video = torch.from_numpy(np.array(result.video.data[0]))
    audio = torch.from_numpy(np.array(result.audio.data[0]))
    audio_sample_rate = result.audio_sample_rate
    print(format_perf_metrics(result.perf_metrics))
else:
    generation_args.update(
        {
            "prompt": prompt_widget.value,
            "generator": torch.Generator("cpu").manual_seed(seed),
            "output_type": "np",
            "return_dict": False,
        }
    )
    if generation_mode == "image-to-video":
        image = load_image(image_widget.value)
        display(image.resize((384, 384 * image.height // image.width)))
        generation_args["image"] = image
        # Disable H.264 recompression of the conditioning image and preserve multi-batch compatibility.
        generation_args["image_crf"] = 0

    videos, audios = pipe(**generation_args)
    video = videos[0]
    audio = audios[0].float().cpu()
    audio_sample_rate = pipe.vocoder.config.output_sampling_rate

output_path = "ltx2.3_output.mp4"
encode_video(video, fps=frame_rate, audio=audio, audio_sample_rate=audio_sample_rate, output_path=output_path)

In [ ]:
from IPython.display import Video

Video(output_path, embed=True)

### Guidance controls

LTX-2.3 supports classifier-free guidance (CFG), spatio-temporal guidance (STG), and modality-isolation guidance, with separate scales for video and audio. Both backends enable all three by default with the same values: `guidance_scale=3.0` and `audio_guidance_scale=7.0`, `stg_scale=1.0` and `audio_stg_scale=1.0` applied to transformer block 28, `modality_scale=3.0` and `audio_modality_scale=3.0`. Video and audio guidance rescaling both default to `0.7` (`guidance_rescale` and `audio_guidance_rescale`). The sliders above and in the demo control the two CFG scales.

STG and modality isolation each add one transformer pass per denoising step, for four passes with the defaults versus two with CFG alone. **Fast mode (CFG only)** sets `stg_scale=0.0, audio_stg_scale=0.0, modality_scale=1.0, audio_modality_scale=1.0` on either backend. This changes guidance and can affect quality; fewer passes do not guarantee a particular end-to-end speedup.

GenAI reports text encoding, transformer inference, video/audio decoding, and total generation times through `result.perf_metrics`. The displayed times are in seconds. Total time also includes work outside the individual inference measurements, such as deferred GPU compilation; Optimum Intel does not expose these GenAI stage metrics.

The Optimum Intel pipeline does not support `use_cross_timestep=True`, and an explicit STG `perturbation_mask` must be uniform across a batch.

## Interactive inference
[back to top](#Table-of-contents:)

The demo exposes the input image only for the image-to-audio-video workflow and returns an MP4 containing the generated audio track. It inherits the notebook's clip and guidance settings. With GenAI, **Stop** requests cancellation at the next denoising callback without discarding the pipeline, and stage timings are shown after a successful generation. Cancellation is not exposed for the Optimum Intel backend.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(pipe, generation_mode, use_genai, generation_args=generation_args, static_config=static_config)

demo.launch(debug=True)
# If you are launching remotely, specify server_name and server_port.
# demo.launch(server_name="your server name", server_port=7860)
# Read more in the docs: https://gradio.app/docs/

## Limitations
[back to top](#Table-of-contents:)

- Export and inference are resource intensive. CPU inference can take a long time.
- The pinned Optimum Intel revision, the matching nightly OpenVINO packages, and the merged NNCF development commit are required for the export and compression paths described in [Prerequisites](#Prerequisites). These updates are not all available in released packages yet.
- FP16 is the reference format. The updated INT4 recipe also compresses the connectors and is experimental here; full-model quality measurements for this recipe are not included.
- The INT8 compression step needs disk space for both the FP16 and INT8 models and enough RAM to hold the FP16 transformer.
- OpenVINO GenAI supports only text-to-audio-video for LTX-2.3. Image-to-audio-video runs with Optimum Intel.
- Text-to-video and image-to-video reuse one text-to-video export with a VAE encoder. Rerun the device/mode, parameter, and pipeline-loading cells after changing the workflow, backend, devices, precision, or static clip dimensions; conversion is not repeated.
- This tutorial covers only the full single-stage `LTX2Pipeline` and `LTX2ImageToVideoPipeline`. Distilled checkpoints, the two-stage latent upsampler, arbitrary conditions, and LoRA adapters are outside its scope. GenAI rejects LoRA adapters for LTX-2 pipelines.
- NPU execution is not exposed until the full audio-video pipeline has been validated on that device.